In [ ]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import requests
import pandas as pd
from time import sleep

all_rows = []

# ── 1. INTERNATIONAL 
url_series = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieInternational/?limit=500"
series_data = requests.get(url_series).json()

for item in series_data['results']:
    uuid = item['uuid']
    commodity_name = item.get('commodity_name', 'Unknown')
    country = item.get('market_name', 'Unknown')
    market = item.get('country_name', 'Unknown')
    price_type = item.get('price_type', 'Unknown')
    unit = item.get('measure_unit_label', 'Unknown')

    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()

    if resp['count'] == 0:
        continue

    datapoints = resp['results'][0]['datapoints']
    rows = []

    for dp in datapoints:
        price_usd = dp.get('price_value_dollar') or dp.get('price_value')
        if price_usd is None:
            continue

        rows.append({
            'date': dp['date'],
            'price_usd': price_usd,
            'price_local': price_usd,   # ← same as price_usd for international
            'currency': 'USD',           # ← dollars
            'commodity_name': commodity_name,
            'country': country,
            'market': market,
            'price_type': price_type,
            'unit': unit,
            'price_source': 'International'
        })

    if not rows:
        continue

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    full_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='MS')
    df = df.set_index('date').reindex(full_range)
    df['commodity_name'] = commodity_name
    df['country'] = country
    df['market'] = market
    df['price_type'] = price_type
    df['unit'] = unit
    df['currency'] = 'USD'
    df['price_source'] = 'International'
    df.index.name = "date"
    df = df.reset_index()
    df = df[['date', 'price_usd', 'price_local', 'currency',
             'commodity_name', 'country', 'market', 'price_type', 'unit', 'price_source']]

    all_rows.append(df)
    print(f"[International] Fetched {commodity_name} ({len(df)} rows)")
    sleep(0.2)

# ── 2. DOMESTIC ───────────────────────────────────────────────────────────────

.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
    if not rows:
        continue

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    full_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='MS')
    df = df.set_index('date').reindex(full_range)
    df['commodity_name'] = commodity_name
    df['country'] = country
    df['market'] = market
    df['price_type'] = price_type
    df['unit'] = unit
    df['currency'] = currency
    df['price_source'] = 'Domestic'
    df.index.name = "date"
    df = df.reset_index()
    df = df[['date', 'price_usd', 'price_local', 'currency',
             'commodity_name', 'country', 'market', 'price_type', 'unit', 'price_source']]

    all_rows.append(df)
    print(f"[Domestic] Fetched {commodity_name} - {market}, {country} ({len(df)} rows)")
    sleep(0.2)

# ── 3. SAVE ───────────────────────────────────────────────────────────────────
if all_rows:
    final_df = pd.concat(all_rows, ignore_index=True)
    final_df.to_csv("all_commodity.csv", index=False)
    print(f"\nSaved all_commodity.csv with {len(final_df)} total rows")
else:
    print("No data fetched.")


[International] Fetched Groundnuts (US Runners 40/50, c.i.f. Rotterdam) (356 rows)
[International] Fetched Cassava Chips (Shipments to China, f.o.b. Koh Sichang) (206 rows)
[International] Fetched Diammonium phosphate, DAP (P fertilizer) (313 rows)
[International] Fetched Dairy: Skim Milk Powder (European & Oceania average indicative export prices, f.o.b) (433 rows)
[International] Fetched Fish oil (Any origin, c.i.f. North West Europe) (356 rows)
[International] Fetched Crude oil (Brent) (313 rows)
[International] Fetched Wheat (CWRS, 13.5%) (313 rows)
[International] Fetched Rice (Super Kernel White Basmati 2%) (313 rows)
[International] Fetched Sorghum (296 rows)
[International] Fetched Palmkernel meal (Expeller pellets, 21/23%, c.i.f. Rotterdam) (356 rows)
[International] Fetched Barley (feed) (308 rows)
[International] Fetched Meat: Pig meat (Meat of swine, fresh, chilled or frozen) (433 rows)
[International] Fetched Dairy: Cheddar Cheese (European & Oceania average indicative exp

C:\Users\morte\AppData\Local\Temp\ipykernel_13020\592497099.py:142: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_rows, ignore_index=True)



Saved all_commodities.csv with 492297 total rows
